# LLM Tool Calling — From Plain Chat to Agents

> **Description:** This notebook builds up tool calling (also called *function calling*) step by step, starting from a plain LLM call and ending with a small generic agent loop.

On its own, an LLM can only answer from what it learned during training plus whatever text you put in the prompt. It cannot look up today's weather, run a calculation it can't do in its head reliably, query a database, or take any action in the real world.

**Tool calling** closes that gap. You describe your own Python functions to the model using a schema. The model then decides *whether* a function is needed and *what arguments* to call it with — but it never runs the function itself. Your code executes it and feeds the result back, and the model uses that result to write its final answer.

## What You'll Learn

| # | Section |
|---|---|
| 1 | A baseline LLM call with no tools |
| 2 | The tool schema contract |
| 3 | A single-tool round trip, end to end |
| 4 | Defining tools with LangChain's `@tool` decorator |
| 5 | Multiple tools — letting the model choose |
| 6 | A generic agent loop, hand-rolled and via `langchain.agents.create_agent` |
| 7 | Controlling tool use with `tool_choice` |

All examples use [LiteLLM](https://docs.litellm.ai/)'s `completion()`, so the same `tools=[...]` schema works no matter which provider is behind the `model` string — see `litellm_benefits.ipynb` in this folder for why that matters.

> **Where this fits:** `mcp_weather_demo/` elsewhere in this repo implements this exact same pattern — a model calling a `get_weather` tool — but through the standardized Model Context Protocol (MCP) instead of a hand-rolled schema. This notebook is the "how tool calling actually works under the hood" companion to that demo.

---

## Setup

Load `OPENAI_API_KEY` from `.env`.

> ⚠️ **Run this notebook from the `part_2_concepts/` folder** — that's what makes the relative `.env` path work, matching the rest of the samples here.

In [1]:
import json
import os

from dotenv import load_dotenv
from litellm import completion

load_dotenv(".env", override=True)

llm_model = "openai/gpt-5.4-nano"

print("OPENAI_API_KEY set:", bool(os.environ.get("OPENAI_API_KEY")))

OPENAI_API_KEY set: True


## 1. Baseline: A Plain LLM Call, No Tools

First, a completely ordinary `completion()` call — no `tools` argument at all. This is the ceiling of what the model can do by itself: reason over its training data and the prompt, nothing else.

Ask it something it cannot possibly know — today's real weather — and watch it have to hedge.

In [2]:
response = completion(
    model=llm_model,
    messages=[{"role": "user", "content": "What is the current weather in Princeton right now?"}],
)
print(response.choices[0].message.content)

I can’t access live weather data from here.  

If you tell me **which Princeton** (NJ or other) and you’re okay with a quick method, I can help you get it right now using your device (e.g., iPhone Weather / Google “weather Princeton”), or you can paste the latest forecast you see and I’ll interpret it.


## 2. The Tool Schema Contract

To let the model request a function call, you describe each function as a JSON object with three parts:

```python
{
    "type": "function",
    "function": {
        "name": "get_weather",                 # exact name your code will dispatch on
        "description": "Get the current ...",  # the ONLY thing telling the model *when* to use this
        "parameters": {                          # standard JSON Schema for the arguments
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name, e.g. Princeton"}
            },
            "required": ["city"],
        },
    },
}
```

A few things worth internalizing:

- The model never executes anything. It only ever replies with *"please call `get_weather` with `{\"city\": \"Princeton\"}`"* — your code has to actually run it.
- `name` and `description` are effectively prompt engineering for functions. The model picks whether and what to call based purely on this text, so vague descriptions lead to wrong or missed tool calls.
- `parameters` is plain [JSON Schema](https://json-schema.org/) — the same `type` / `properties` / `required` vocabulary you may already know from API request validation.

The next section wires this schema into a real, executable round trip.

## 3. A Single Tool, End to End

A tool-calling round trip always has the same four steps:

1. Send the user's message plus your `tools` schema.
2. The model replies with a `tool_calls` request instead of a normal answer.
3. Your code runs the real Python function and gets a result.
4. You send that result back as a new `role: "tool"` message, and the model produces the final natural-language answer.

Step 1 and 2 first:

In [3]:
def get_weather(city: str) -> str:
    weather_data = {
        "Princeton": "Sunny, 75F",
        "Seattle": "Cloudy, 60F",
        "Austin": "Sunny, 85F",
    }
    return weather_data.get(city, "Weather information unavailable")


weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather for a specific city.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string", "description": "City name, e.g. Princeton"}},
            "required": ["city"],
        },
    },
}

messages = [{"role": "user", "content": "What is the current weather in Princeton?"}]

response = completion(model=llm_model, messages=messages, tools=[weather_tool])
response_message = response.choices[0].message

print("content:", response_message.content)
print("tool_calls:", response_message.tool_calls)

content: None
tool_calls: [ChatCompletionMessageToolCall(function=Function(arguments='{"city":"Princeton"}', name='get_weather'), id='call_bUDAcVueWKXs1Fz7yMZ30lyQ', type='function')]


Notice `content` is empty — the model didn't answer the question, it asked *us* to call `get_weather(city="Princeton")`. Steps 3 and 4 finish the round trip:

In [4]:
tool_call = response_message.tool_calls[0]
args = json.loads(tool_call.function.arguments)
result = get_weather(**args)
print("Tool result:", result)

messages.append(response_message)
messages.append(
    {
        "tool_call_id": tool_call.id,
        "role": "tool",
        "name": tool_call.function.name,
        "content": result,
    }
)

final_response = completion(model=llm_model, messages=messages, tools=[weather_tool])
print("\nFinal answer:", final_response.choices[0].message.content)

Tool result: Sunny, 75F

Final answer: The current weather in **Princeton** is **Sunny, 75°F**.


## 4. Defining Tools with Decorator Syntax (LangChain)

Hand-writing a JSON Schema dict for every function works, but it duplicates information you already wrote once: the parameter names and types are right there in the function signature, and a good docstring already explains what the function does.

[LangChain](https://python.langchain.com/) ships a `@tool` decorator that turns a plain function into a `StructuredTool` by reading its type hints and docstring — the same idea FastMCP's `@mcp.tool()` (used in `mcp_weather_demo/server/weather_server.py` in this repo) and the OpenAI Agents SDK's `@function_tool` are built on.

Because a `StructuredTool` already knows its own schema, LangChain's chat models can consume it directly via `model.bind_tools([...])` — no manual conversion needed. The example below therefore calls the LLM through LangChain end to end (`ChatOpenAI` + `bind_tools`), rather than through `litellm.completion()` as earlier sections did.

`langchain_core.utils.function_calling.convert_to_openai_tool()` can still turn a LangChain tool into the plain `{"type": "function", "function": {...}}` schema — the same shape as `weather_tool` from section 3, just generated automatically instead of hand-written. Section 5 puts `get_current_time` to work alongside `get_weather`.

In [5]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI


@tool
def get_current_time(timezone: str) -> str:
    """Get the current time in a given IANA timezone, e.g. 'America/New_York'."""
    from datetime import datetime
    from zoneinfo import ZoneInfo

    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M:%S %Z")


# Call the LLM through LangChain itself: bind the tool straight to a chat model.
chat_model = ChatOpenAI(model="gpt-5.4-nano")
chat_model_with_tools = chat_model.bind_tools([get_current_time])

response = chat_model_with_tools.invoke("What time is it in Tokyo?")
tool_call = response.tool_calls[0]
print("model picked:", f"{tool_call['name']}({tool_call['args']})")
print("actual result:", get_current_time.invoke(tool_call["args"]))

model picked: get_current_time({'timezone': 'Asia/Tokyo'})
actual result: 2026-07-26 03:18:04 JST


## 5. Multiple Tools — Letting the Model Choose

Real assistants offer several tools at once and let the model pick the right one per question. `bind_tools()` doesn't care whether a tool's schema was hand-written or decorator-generated, so `get_weather` (the hand-written tool from section 3) and `get_current_time` (the LangChain `StructuredTool` from section 4) can be bound together as-is — no conversion needed for either.

In [6]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI


@tool
def get_current_time(timezone: str) -> str:
    """Get the current time in a given IANA timezone, e.g. 'America/New_York'."""
    from datetime import datetime
    from zoneinfo import ZoneInfo

    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M:%S %Z")

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a specific city."""
    weather_data = {
        "Princeton": "Sunny, 75F",
        "Seattle": "Cloudy, 60F",
        "Austin": "Sunny, 85F",
    }
    return weather_data.get(city, "Weather information unavailable")

tools = [get_weather, get_current_time]
chat_model = ChatOpenAI(model="gpt-5.4-nano")
chat_model_with_tools = chat_model.bind_tools(tools)

for question in ["What's the weather in Seattle?", "What time is it in Tokyo?"]:
    response = chat_model_with_tools.invoke(question)
    tool_call = response.tool_calls[0]
    print(f"Q: {question}")
    print(f"-> model picked: {tool_call['name']}({tool_call['args']})")
    print()

Q: What's the weather in Seattle?
-> model picked: get_weather({'city': 'Seattle'})

Q: What time is it in Tokyo?
-> model picked: get_current_time({'timezone': 'Asia/Tokyo'})



## 6. The Agent Loop

Sections 3 and 5 handled one tool call by hand. A real agent just wraps that pattern in a loop: keep calling the model and executing whatever it asks for, until it finally responds with plain text instead of a tool request.

This also naturally handles **parallel tool calls** — a single assistant turn can request more than one function at once, so the loop iterates over `response.tool_calls` rather than assuming there's exactly one.

This section calls the LLM through LangChain (`chat_model.bind_tools(...)` from section 4) instead of `litellm.completion()`, reusing the same `tools` list from section 5.

In [7]:
def run_agent_loop(messages, tools, max_iterations=5):
    tools_by_name = {tool.name: tool for tool in tools}
    response = None

    for _ in range(max_iterations):
        response = chat_model_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            break

        for tool_call in response.tool_calls:
            selected_tool = tools_by_name[tool_call["name"]]
            result = selected_tool.invoke(tool_call["args"])
            messages.append(
                {"role": "tool", "tool_call_id": tool_call["id"], "content": str(result)}
            )

    return response

tools = [get_weather, get_current_time]
chat_model = ChatOpenAI(model="gpt-5.4-nano")
chat_model_with_tools = chat_model.bind_tools(tools)

messages = [{"role": "user", "content": "What's the weather in Austin, and what time is it in Tokyo?"}]
final_response = run_agent_loop(messages, tools)
print(final_response.content)

- **Austin weather:** Sunny, **85°F**
- **Time in Tokyo:** **2026-07-26 03:23:31 JST**


### The Same Loop, Built In: `langchain.agents.create_agent`

`run_agent_loop` above is a small, hand-rolled version of something LangChain already ships. `create_agent` takes a model (a string like `"openai:gpt-5.4-nano"`, or a chat model instance) plus a list of tools, and returns a compiled [LangGraph](https://langchain-ai.github.io/langgraph/) graph that runs the same request → execute tool → respond cycle internally — parallel tool calls, `role: "tool"` message bookkeeping, and the stop condition all included.

Invoke it with `{"messages": [...]}` instead of a plain list. The result is the full conversation state; `result["messages"][-1].content` is the final answer, and everything in between shows the same `AIMessage` → `ToolMessage` → `AIMessage` shape the manual loop produced.

In [9]:
from langchain.agents import create_agent

tools = [get_weather, get_current_time]
agent = create_agent("openai:gpt-5.4-nano", tools=tools)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in Austin, and what time is it in Tokyo?"}]}
)

# json_result = json.dumps(result, default=str, indent=4)
# print(json_result)

print("Final answer:", result["messages"][-1].content)

Final answer: - **Austin weather:** Sunny, **85°F**
- **Time in Tokyo:** **2026-07-26 03:27:56 JST**


## 7. Controlling Tool Use with `tool_choice`

`tool_choice` overrides the model's default judgment. LangChain exposes it as a `bind_tools()` keyword argument:

- unset / `"auto"` (the default) — model decides whether to call anything.
- `"none"` — ignore the tools entirely, even though they're provided.
- a tool's name as a string, e.g. `"get_weather"` — force that specific tool, regardless of the question.

Forcing a mismatched tool is a good way to see this isn't optional guidance — it's a hard constraint:

In [10]:
forced = chat_model.bind_tools(tools, tool_choice="get_weather")
forced_response = forced.invoke("Tell me a fun fact about the ocean.")
print("tool_choice=<forced get_weather> ->", forced_response.tool_calls)

disabled = chat_model.bind_tools(tools, tool_choice="none")
disabled_response = disabled.invoke("What's the weather in Princeton?")
print("\ntool_choice='none' ->", disabled_response.content)

tool_choice=<forced get_weather> -> [{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'call_6tfIZca9dsuPwYIJF9OCnFzu', 'type': 'tool_call'}]

tool_choice='none' -> I can’t access live weather data right now, so I can’t tell you the current conditions in Princeton.  

If you tell me which Princeton you mean (e.g., **Princeton, NJ** or **Princeton, MA**) and whether you want **right now** or a **forecast for today/tomorrow**, I can help you find the most relevant source—or estimate what to expect based on typical seasonal patterns.


## Summary

| Concept | What it does |
|---|---|
| `tools=[...]` | Describes your Python functions to the model as JSON Schema |
| LangChain's `@tool` decorator | Wraps a function into a `StructuredTool` from its signature and docstring; `convert_to_openai_tool()` turns it into the same schema shape |
| `message.tool_calls` | The model's request to call one or more functions, with arguments |
| `role: "tool"` message | How you send a function's result back into the conversation |
| Agent loop | Repeats request → execute → respond until the model gives a plain answer |
| `langchain.agents.create_agent` | Same loop, prebuilt on LangGraph — pass a model + tools, invoke with `{"messages": [...]}` |
| `tool_choice` | Forces, forbids, or defaults (`"auto"`) tool use |

**Practices worth keeping:**

- Write tool `description`s as carefully as you'd write documentation for a teammate — the model has nothing else to go on.
- Never blindly execute model-supplied arguments (e.g. via `eval()`); validate them the same way you would any other untrusted input.
- Always cap agent loops with a `max_iterations` guard so a confused model can't loop forever.
- Whether a schema is hand-written or built with LangChain's decorator makes no difference to the model — pick whichever keeps your codebase easiest to maintain as the number of tools grows, and normalize how each tool is *executed* (plain call vs. `.invoke()`) in one place, like `available_functions` does above.

See `mcp_weather_demo/` in this repo for the same `get_weather` tool exposed through the standardized Model Context Protocol instead of an inline schema — useful once you want tools shared across multiple clients or written in a different language.